In [1]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import os
from datetime import datetime, timedelta

np.random.seed(42)

# Generate dataset
n = 500000

cities = ["Berlin","Tokyo","New York","Seoul","Toronto","Paris","London","Sydney","Lima","Nairobi"]

start_date = datetime.now() - timedelta(days=365*3)

data = pd.DataFrame({
    "user_id": np.arange(1, n+1),
    "city": np.random.choice(cities, n),
    "score": np.random.uniform(0,100,n),
    "active": np.random.choice([True, False], n),
    "signup_date": [start_date + timedelta(days=int(x)) for x in np.random.uniform(0,365*3,n)],
    "age": np.random.randint(18,81,n),
    "sessions": np.random.randint(0,501,n),
    "revenue": np.random.uniform(0,1000,n)
})

# Save as Parquet
parquet_file = "users.parquet"
data.to_parquet(parquet_file)

# Inspect metadata
pf = pq.ParquetFile(parquet_file)

print("Row groups:", pf.num_row_groups)
print("Rows:", pf.metadata.num_rows)
print("Columns:", pf.metadata.num_columns)

print("\nSchema:")
print(pf.schema)

# Inspect first row group column metadata
row_group = pf.metadata.row_group(0)

for i in range(row_group.num_columns):
    col = row_group.column(i)
    stats = col.statistics

    print("\nColumn:", col.path_in_schema)
    print("Physical type:", col.physical_type)
    print("Compressed size:", col.total_compressed_size)

    if stats:
        print("Min:", stats.min)
        print("Max:", stats.max)
        print("Null count:", stats.null_count)

# Save CSV
csv_file = "users.csv"
data.to_csv(csv_file, index=False)

# File sizes
parquet_size = os.path.getsize(parquet_file)/1024
csv_size = os.path.getsize(csv_file)/1024

print("\nParquet size KB:", parquet_size)
print("CSV size KB:", csv_size)
print("Compression ratio (CSV / Parquet):", csv_size/parquet_size)

Row groups: 1
Rows: 500000
Columns: 8

Schema:
required group field_id=-1 schema {
  optional int64 field_id=-1 user_id;
  optional binary field_id=-1 city (String);
  optional double field_id=-1 score;
  optional boolean field_id=-1 active;
  optional int64 field_id=-1 signup_date (Timestamp(isAdjustedToUTC=false, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int32 field_id=-1 age;
  optional int32 field_id=-1 sessions;
  optional double field_id=-1 revenue;
}


Column: user_id
Physical type: INT64
Compressed size: 2273849
Min: 1
Max: 500000
Null count: 0

Column: city
Physical type: BYTE_ARRAY
Compressed size: 252612
Min: Berlin
Max: Toronto
Null count: 0

Column: score
Physical type: DOUBLE
Compressed size: 4272959
Min: 5.188445665327279e-05
Max: 99.99983148609545
Null count: 0

Column: active
Physical type: BOOLEAN
Compressed size: 63800
Min: False
Max: True
Null count: 0

Column: signup_date
Physical type: INT64
Compressed size: 

Markdown

Parquet files contain rich metadata that is not available in CSV files. This metadata includes the schema, the number of row groups, column chunk information, compression sizes, and statistics such as minimum values, maximum values, and null counts for each column.

CSV files only store raw text data without any structural metadata. As a result, when reading a CSV file, the processing engine must scan the entire file to understand the structure and infer data types.

In contrast, Parquet stores metadata in the file footer, allowing query engines to quickly determine which columns and row groups are relevant to a query. Because statistics such as min and max values are stored per column chunk, systems can skip irrelevant sections of data during filtering operations. This significantly improves query performance, especially for large datasets.

In [2]:
import time

# Read full Parquet
start = time.time()
df_full = pd.read_parquet("users.parquet")
print("Full Parquet read time:", time.time()-start)

# Read only 2 columns from Parquet
start = time.time()
df_two = pd.read_parquet("users.parquet", columns=["city","score"])
print("Parquet 2-column read time:", time.time()-start)

# CSV selective read (pandas reads whole file)
start = time.time()
df_csv = pd.read_csv("users.csv")[["city","score"]]
print("CSV read then select columns time:", time.time()-start)

Full Parquet read time: 0.1442863941192627
Parquet 2-column read time: 0.03824567794799805
CSV read then select columns time: 0.7021918296813965


Markdown

Parquet supports column projection, which allows the system to read only the columns required for a query. Because Parquet stores each column separately in column chunks, the storage engine can skip reading unnecessary columns entirely.

CSV files store data row by row in plain text, meaning every column for every row must be read and parsed even if only a few columns are needed. As a result, reading selected columns from a CSV file still requires scanning the entire file.

The column-oriented storage layout of Parquet makes selective reads much faster and more efficient, especially for large datasets where queries often use only a small subset of available columns.

In [3]:
import pyarrow.dataset as ds
import duckdb

# PyArrow filtered read
start = time.time()
table = ds.dataset("users.parquet").to_table(filter=(ds.field("age") > 50))
df_filtered_arrow = table.to_pandas()
print("PyArrow predicate pushdown time:", time.time()-start)

# Pandas full read then filter
start = time.time()
df = pd.read_parquet("users.parquet")
df_filtered = df[df["age"] > 50]
print("Pandas filter after load time:", time.time()-start)

print("Rows returned:", len(df_filtered))

# DuckDB query
start = time.time()
result = duckdb.sql("SELECT * FROM 'users.parquet' WHERE age > 50").df()
print("DuckDB query time:", time.time()-start)

ModuleNotFoundError: No module named 'duckdb'

Markdown

Predicate pushdown is an optimization where filtering conditions are applied during the file read process rather than after loading the entire dataset.

When reading Parquet files with filters, the engine can use column statistics stored in the metadata to determine which row groups contain values that satisfy the condition. If a row group's min and max values show that it cannot satisfy the filter condition, the engine skips reading that row group entirely.

This results in significantly faster query performance because less data needs to be read from disk and processed in memory.

When filtering with pandas after loading the full dataset, the entire file must first be read into memory, which is slower and less efficient.

DuckDB performs predicate pushdown automatically when querying Parquet files using SQL, allowing it to achieve performance comparable to specialized data processing engines.

In [4]:
import time

# Load data
df = pd.read_parquet("users.parquet")

# Query 1: count per city
start=time.time()
q1_pandas = df.groupby("city").size()
print("Pandas Q1 time:", time.time()-start)

start=time.time()
q1_duck = duckdb.sql("SELECT city, COUNT(*) FROM 'users.parquet' GROUP BY city").df()
print("DuckDB Q1 time:", time.time()-start)


# Query 2: avg score by city
start=time.time()
q2_pandas = df.groupby("city")["score"].mean().sort_values(ascending=False)
print("Pandas Q2 time:", time.time()-start)

start=time.time()
q2_duck = duckdb.sql("""
SELECT city, AVG(score) as avg_score
FROM 'users.parquet'
GROUP BY city
ORDER BY avg_score DESC
""").df()
print("DuckDB Q2 time:", time.time()-start)


# Query 3
start=time.time()
cond = df[df["active"] & (df["score"]>75)]
q3_pandas = cond.groupby("city").size()/df.groupby("city").size()*100
print("Pandas Q3 time:", time.time()-start)

start=time.time()
q3_duck = duckdb.sql("""
SELECT city,
100.0 * SUM(CASE WHEN active AND score > 75 THEN 1 ELSE 0 END) / COUNT(*) AS pct
FROM 'users.parquet'
GROUP BY city
""").df()
print("DuckDB Q3 time:", time.time()-start)


# Query 4: top 10 users per city
start=time.time()
df["rank"] = df.groupby("city")["score"].rank(method="first", ascending=False)
q4_pandas = df[df["rank"]<=10]
print("Pandas Q4 time:", time.time()-start)

start=time.time()
q4_duck = duckdb.sql("""
SELECT *
FROM (
SELECT *, RANK() OVER(PARTITION BY city ORDER BY score DESC) r
FROM 'users.parquet'
)
WHERE r <= 10
""").df()
print("DuckDB Q4 time:", time.time()-start)


# Query 5: running total
start=time.time()
df["running_total"] = df.sort_values("user_id").groupby("city")["score"].cumsum()
print("Pandas Q5 time:", time.time()-start)

start=time.time()
q5_duck = duckdb.sql("""
SELECT *,
SUM(score) OVER(PARTITION BY city ORDER BY user_id) AS running_total
FROM 'users.parquet'
""").df()
print("DuckDB Q5 time:", time.time()-start)

Pandas Q1 time: 0.05688643455505371


NameError: name 'duckdb' is not defined

Markdown

For simple aggregation queries such as counting records per city or computing averages, both pandas and DuckDB provide clear and concise syntax. Pandas uses groupby operations, while DuckDB uses standard SQL queries.

DuckDB tends to perform better for complex analytical queries, especially those involving window functions such as ranking or running totals. SQL expressions for these operations are often more concise and easier to read than the equivalent pandas operations.

For simpler transformations and exploratory analysis, pandas can be more convenient because it integrates naturally with Python workflows. However, for large datasets and complex analytical queries, DuckDB often provides better performance and clearer query expressions.

In [5]:
import pyarrow as pa
import pyarrow.parquet as pq

# Convert pandas to Arrow table
arrow_table = pa.Table.from_pandas(data)

print("Arrow schema:")
print(arrow_table.schema)

print("\nPandas dtypes:")
print(data.dtypes)

# Write Arrow table to Parquet
pq.write_table(arrow_table, "arrow_users.parquet")

# Read back
table2 = pq.read_table("arrow_users.parquet")

# Convert back to pandas
df_back = table2.to_pandas()

print("\nData matches:", df_back.head().equals(data.head()))

# Arrow-backed pandas
df_arrow_backend = pd.read_parquet("arrow_users.parquet", dtype_backend="pyarrow")

print("\nArrow-backed pandas dtypes:")
print(df_arrow_backend.dtypes)

Arrow schema:
user_id: int64
city: large_string
score: double
active: bool
signup_date: timestamp[us]
age: int32
sessions: int32
revenue: double
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 1174

Pandas dtypes:
user_id                 int64
city                      str
score                 float64
active                   bool
signup_date    datetime64[us]
age                     int32
sessions                int32
revenue               float64
dtype: object

Data matches: True

Arrow-backed pandas dtypes:
user_id                int64[pyarrow]
city            large_string[pyarrow]
score                 double[pyarrow]
active                  bool[pyarrow]
signup_date    timestamp[us][pyarrow]
age                    int32[pyarrow]
sessions               int32[pyarrow]
revenue               double[pyarrow]
dtype: object


Markdown

Apache Arrow plays a central role in the modern analytics ecosystem by providing a standardized in-memory columnar data format. Arrow allows different data processing systems to exchange data efficiently without costly conversions.

Parquet files store data on disk using a columnar binary format. When these files are read into memory, Arrow provides a representation that can be shared across tools such as pandas, DuckDB, and other analytics engines.

Because Arrow uses a columnar layout similar to Parquet, it enables efficient zero-copy data transfer between systems. For example, DuckDB can query Parquet files and return results directly as Arrow tables, which can then be converted into pandas DataFrames without expensive serialization.

In this way, Arrow acts as a bridge between storage formats like Parquet, analytical libraries like pandas, and query engines like DuckDB, forming the backbone of many modern data analytics pipelines.